### Imports

In [2]:
import copy
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

ROOT = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / "src" / "paths.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Could not locate project root containing src/paths.py")
sys.path.insert(0, str(ROOT))

from src.paths import CONFIG, PROCESSED, SENSITIVITY
from src.bess_optimization import optimize_bess


### Load inputs

In [3]:
profile_options = [
    PROCESSED / "optimization_profile.csv",
    PROCESSED / "system_hourly_profile.csv",
]
profile_path = next((path for path in profile_options if path.is_file()), None)
candidates_path = PROCESSED / "candidate_bess_sites.csv"
profile = pd.read_csv(profile_path) if profile_path else pd.DataFrame()
candidates = pd.read_csv(candidates_path) if candidates_path.is_file() else pd.DataFrame()
if not profile.empty and "timestamp" in profile.columns:
    profile["timestamp"] = pd.to_datetime(profile["timestamp"], errors="coerce", utc=True)
config = json.loads((CONFIG / "model_config.json").read_text(encoding="utf-8"))
inputs_ready = not profile.empty and not candidates.empty
if not inputs_ready:
    print("Sensitivity analysis inputs are not ready.")
    if profile_path is None:
        print("Provide datasets/processed/optimization_profile.csv (or system_hourly_profile.csv).")
    if not candidates_path.is_file() or candidates.empty:
        print("Run notebook/06_BESS_Candidate_Site_Selection.ipynb after loading grid data.")
else:
    print("Profile records:", len(profile))
    print("Candidate substations:", len(candidates))


Sensitivity analysis inputs are not ready.
Provide datasets/processed/optimization_profile.csv (or system_hourly_profile.csv).
Run notebook/06_BESS_Candidate_Site_Selection.ipynb after loading grid data.


### Create sensitivity scenarios

In [4]:
scenarios = [
    {
        "scenario": "Low battery cost",
        "cost_multiplier": 0.80,
        "curtailment_penalty": 40
    },
    {
        "scenario": "Base case",
        "cost_multiplier": 1.00,
        "curtailment_penalty": 40
    },
    {
        "scenario": "High battery cost",
        "cost_multiplier": 1.20,
        "curtailment_penalty": 40
    },
    {
        "scenario": "High curtailment penalty",
        "cost_multiplier": 1.00,
        "curtailment_penalty": 80
    }
]

### Run sensitivity analysis

In [5]:
if inputs_ready:
    sensitivity_results = []

    for scenario in scenarios:
        scenario_config = copy.deepcopy(config)
        scenario_config["battery"]["power_cost_usd_per_kw"] *= scenario["cost_multiplier"]
        scenario_config["battery"]["energy_cost_usd_per_kwh"] *= scenario["cost_multiplier"]
        scenario_config["optimization"]["curtailment_penalty_usd_per_mwh"] = scenario["curtailment_penalty"]

        result = optimize_bess(profile, candidates, scenario_config)
        sites = result["selected_sites"]
        sensitivity_results.append({
            "scenario": scenario["scenario"],
            "objective_value": result["objective_value"],
            "number_of_sites": len(sites),
            "total_power_mw": sites["power_mw"].sum() if not sites.empty else 0,
            "total_energy_mwh": sites["energy_mwh"].sum() if not sites.empty else 0,
        })

    sensitivity_df = pd.DataFrame(sensitivity_results)
    display(sensitivity_df)
else:
    sensitivity_df = pd.DataFrame(columns=["scenario", "objective_value", "number_of_sites", "total_power_mw", "total_energy_mwh"])
    print("Sensitivity analysis skipped until the required inputs are populated.")


Sensitivity analysis skipped until the required inputs are populated.


### Visualize sensitivity analysis

In [6]:
if not sensitivity_df.empty:
    plt.figure(figsize=(11, 5))
    plt.bar(sensitivity_df["scenario"], sensitivity_df["total_energy_mwh"])
    plt.title("Battery Energy Capacity Under Different Scenarios")
    plt.xlabel("Scenario")
    plt.ylabel("Battery Energy Capacity (MWh)")
    plt.xticks(rotation=25)
    plt.tight_layout()
    plt.savefig(SENSITIVITY / "battery_sensitivity.png", dpi=300)
    plt.show()
else:
    print("Sensitivity chart is unavailable until the analysis runs.")


Sensitivity chart is unavailable until the analysis runs.


### Save results

In [7]:
if sensitivity_df.empty:
    print("No sensitivity results to save.")
else:
    SENSITIVITY.mkdir(parents=True, exist_ok=True)
    sensitivity_df.to_csv(
        SENSITIVITY / "sensitivity_results.csv",
        index=False
    )

No sensitivity results to save.
